## Decision Tree Splitting

1. Gini impurity of one group: $Gini=1-\sum p_c^2$. Binary: $1-(p^2+(1-p)^2)$, lower = purer. 50/50 mix: 0.5 (max for 2 classes). 90/10 mix: 0.18. Pure: 0. General $k$-class max $=1-1/k$.
2. Scoring a candidate split, weighted Gini: $weighted\_gini=\frac{n_L}{n}Gini(L)+\frac{n_R}{n}Gini(R)$, weighting by group size stops a split that carves off one tiny pure sliver from being rewarded. Best split = lowest weighted Gini.
3. Candidate thresholds: midpoints between consecutive sorted unique values, avoids a threshold landing exactly on an observed value, which side would that fall on?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 100

# XOR: no straight line separates this; only x AND y together carry signal
# class 0: top-right and bottom-left quadrants
c0a = rng.normal(loc=[2, 2], scale=0.5, size=(n, 2))
c0b = rng.normal(loc=[-2, -2], scale=0.5, size=(n, 2))
# class 1: top-left and bottom-right quadrants
c1a = rng.normal(loc=[-2, 2], scale=0.5, size=(n, 2))
c1b = rng.normal(loc=[2, -2], scale=0.5, size=(n, 2))

X_tree = np.vstack([c0a, c0b, c1a, c1b])
y_tree = np.hstack([np.zeros(2*n), np.ones(2*n)])

plt.scatter(X_tree[:, 0], X_tree[:, 1], c=y_tree, cmap="coolwarm", alpha=0.6)
plt.title("XOR-pattern toy data, no straight line can separate this")
plt.show()

In [ ]:
# Gini = 1 - sum(p_c^2); binary, so mean(y) = p(class 1)
def gini(y):
    if len(y) == 0:
        return 0
    p = np.mean(y)
    return 1 - (p**2 + (1 - p)**2)

print("gini of pure group [1,1,1,1]:", gini(np.array([1,1,1,1])))   # expect 0
print("gini of 50/50 group [1,1,0,0]:", gini(np.array([1,1,0,0])))  # expect 0.5
print("gini of full y_tree (50/50 by construction):", gini(y_tree)) # expect 0.5


In [ ]:
# tries every (feature, threshold) candidate, scores with weighted gini, keeps the best
def best_split(X, y):
    best_gini = float("inf")
    best_feature, best_threshold = None, None
    n_total = len(y)

    for feature_idx in range(X.shape[1]):
        values = np.sort(np.unique(X[:, feature_idx]))  # np.sort redundant, unique already sorts
        thresholds = (values[:-1] + values[1:]) / 2   # midpoints, avoids landing on a data point

        for t in thresholds:
            left_mask = X[:, feature_idx] <= t
            y_left, y_right = y[left_mask], y[~left_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            weighted_gini = (len(y_left)/n_total) * gini(y_left) + (len(y_right)/n_total) * gini(y_right)

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature, best_threshold = feature_idx, t

    return best_feature, best_threshold, best_gini

feature, threshold, g = best_split(X_tree, y_tree)
print(f"best split: feature={feature}, threshold={threshold:.3f}, weighted gini={g:.3f}")
# ~0.496, barely better than 0.5 -- true XOR has no informative single split


In [ ]:
left_mask = X_tree[:, feature] <= threshold
X_left, y_left = X_tree[left_mask], y_tree[left_mask]
X_right, y_right = X_tree[~left_mask], y_tree[~left_mask]

print("left: n =", len(y_left), ", class balance =", y_left.mean(), ", gini =", gini(y_left))
print("right: n =", len(y_right), ", class balance =", y_right.mean(), ", gini =", gini(y_right))

f_left, t_left, g_left = best_split(X_left, y_left)
f_right, t_right, g_right = best_split(X_right, y_right)
print(f"left's best next split: feature={f_left}, threshold={t_left:.3f}, weighted gini={g_left:.3f}")
print(f"right's best next split: feature={f_right}, threshold={t_right:.3f}, weighted gini={g_right:.3f}")

# right = 12 pts, barely skewed -- noise, not signal. left still ~unsolved (gini 0.5)
# greedy trees have no lookahead -- can't solve pure XOR in 2 splits, ties broken by noise
# real fraud interactions aren't this adversarial: amt alone already carried real signal


#### Entropy, the alternative split criterion

Formula: entropy = -sum(p_c * log2(p_c)). Same job as Gini (lower = purer node), different formula, different curve shape. Worked comparison, 50/50 split: Gini = 0.5 (from section 1 above), entropy = -(0.5*log2(0.5) + 0.5*log2(0.5)) = -(0.5*-1 + 0.5*-1) = 1.0 (max entropy for 2 classes, matching `probability-statistics.ipynb`/`loss-functions.ipynb`'s entropy=uncertainty framing exactly, a pure node has entropy 0, no uncertainty).

Practical difference: entropy is slightly more computationally expensive (a log call vs a square), and tends to produce marginally more balanced trees since its curve is more sharply peaked at 50/50 than Gini's. In practice the two rarely produce meaningfully different trees, Gini is the more common default (sklearn's default) mainly because it's cheaper to compute at scale, not because it makes better splits.

In [ ]:
def entropy(y):
    if len(y) == 0:
        return 0
    p = np.mean(y)
    if p == 0 or p == 1:
        return 0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

for label, y_toy in [("pure", np.array([1,1,1,1])), ("50/50", np.array([1,1,0,0])), ("90/10", np.array([1]*9+[0]))]:
    print(f"{label}: gini={gini(y_toy):.3f}, entropy={entropy(y_toy):.3f}")

#### Overfitting, max_depth, and pruning

An unconstrained tree keeps splitting until every leaf is perfectly pure (or has 1 sample), that means memorizing the training set, a leaf with exactly 1 training point generalizes to nothing. Worked illustration: the XOR toy above needed a second split (left/right) to make progress, an unconstrained tree would keep going, splitting smaller and smaller noise-driven slivers, chasing gini improvements that come from overfitting to individual points rather than real structure, exactly what the comment in the code above ("right = 12 pts, barely skewed, noise not signal") already flagged happening after just 2 splits.

Controls, same complexity-penalty family as `regularization.ipynb`'s tree-specific section:
```
max_depth: hard cap on how many splits deep any path can go
min_samples_split: a node needs at least this many samples to be split at all
min_samples_leaf: a split is rejected if either resulting leaf would have fewer samples than this
```
Pruning (post-hoc, the alternative to capping depth upfront): grow the full tree, then remove splits that don't improve validation performance enough, letting the tree grow past where you'd cap it, then trimming back based on evidence, rather than blocking growth by a fixed rule that might cut off a genuinely useful split too early in one branch while allowing an unnecessary one in another.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

unconstrained = DecisionTreeClassifier(random_state=42)
unconstrained.fit(X_tree, y_tree)
print("unconstrained tree depth:", unconstrained.get_depth(), "| leaves:", unconstrained.get_n_leaves())
print("unconstrained train accuracy:", unconstrained.score(X_tree, y_tree))

capped = DecisionTreeClassifier(max_depth=2, random_state=42)
capped.fit(X_tree, y_tree)
print("\nmax_depth=2 tree accuracy:", capped.score(X_tree, y_tree), "(matches the ~0.5-ish ceiling the hand-worked 2-split search above found, XOR genuinely needs depth to solve)")

#### Regression trees

Same split-search algorithm, different impurity measure: instead of Gini/entropy (which need discrete classes), regression trees minimize variance (or equivalently, MSE, see `loss-functions.ipynb`) within each resulting leaf. Leaf prediction = the MEAN of the training targets that land in that leaf, not a class vote. Everything else, greedy threshold search, max_depth, pruning, works identically, just swap the impurity function.

#### Assumptions and limitations

Axis-aligned splits only: every split is "feature_i <= threshold," a single straight cut perpendicular to one axis. A genuinely DIAGONAL decision boundary needs many small axis-aligned steps to approximate (a staircase pattern), never a single clean cut, this is a real geometric limitation, not just a tuning issue.

Greedy, no lookahead: the split search above picks the single best split available RIGHT NOW, it never considers "this split looks locally weak but sets up a much better split 2 levels down." This is exactly why the XOR toy above defeats a single tree so badly, weighted gini ~0.496 is barely better than the unsplit 0.5, greedy search cannot see that combining BOTH features perfectly separates the classes, it can only evaluate one feature-threshold at a time.

No extrapolation: leaf predictions are literally training-data averages/majorities, a regression tree can never predict a value outside the range seen in training, unlike linear regression's formula which extrapolates freely (for better or worse).

No feature-scaling requirement: unlike SVM/KNN/logreg (see `classical-ml.ipynb`), splits compare a feature against its OWN threshold, scale never affects which split gets chosen.

This is precisely why ensembles exist: a single tree's greedy, axis-aligned, no-lookahead limitations are real, but averaging many independent trees over different data/feature subsets (`bagging.ipynb`) or sequentially correcting a tree's specific mistakes with the next tree (`boosting.ipynb`) both directly compensate for a single tree's weaknesses, without changing the core split-search algorithm shown here at all.

#### Feature importance

Computed from how much each feature's splits reduced impurity (Gini/entropy/variance), summed across every node that split on that feature, weighted by how many samples passed through that node. A feature used high up in the tree, splitting many samples with a big Gini reduction, gets high importance, a feature used only in one small deep leaf gets low importance even if that one split was locally perfect.

In [ ]:
print("feature importances (unconstrained tree):", unconstrained.feature_importances_)